In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# This script processes the VX data and creates a long-type dataframe.
#  It reads VX data from a CSV file, transforms it into a long format,
#  and saves the result to a new CSV file.
# 26.01.07 세종_문어에 대해서만 완료. 

import pandas as pd
import numpy as np
from typing import Tuple, List

import sys
from pathlib import Path
ROOT = Path.cwd().parents[0]   # 상위폴더를 ROOT로 넣음.
sys.path.append(str(ROOT))
from stats.en_no_fix import fix_en_columns #EN_No 수정

In [ ]:
CSV_PATH = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver7.csv"
df = pd.read_csv(CSV_PATH)

In [ ]:
# ①설정 & 헬퍼 함수설정 & 헬퍼 함수

KEEP_COLS = [
    "ID", "docu_id", "category", "sen_id", "word_id2", "form/label",
    "N_form", "N_label", "V_form", "V_label",
    "EP_form", "EP_label", "EN_form", "EN_label", "J_form", "J_label",
    "EN_No", "V_No"
]

OUT_COLS_ADDED = [
    "Next_VX_No", "Next_VX_No_form",
    "vx0_No", "vx0_form",
    "vx0_order",
    "vx_len",
    "V_word_id",
    "f_word_id"
]

def norm_empty(series):
    return series.fillna("").astype(str).str.strip()

def make_vx_form_two_row(en_form, en_j_form, n_form_next, v_form_next):
    s = en_form
    if en_j_form:
        s += en_j_form
    if n_form_next:
        s += " " + n_form_next
    s += " " + v_form_next
    return s.strip()

def make_vx_form_three_row(en_form, n_form_mid, j_form_mid, v_form_last):
    s = en_form + " " + n_form_mid
    if j_form_mid:
        s += j_form_mid
    s += " " + v_form_last
    return s.strip()

#②: VX edge 생성 함수
def build_vx_edges_word_only(df_word, df_label):
    df = df_word.copy()
    lab = df_label.copy()

    for c in ["sen_id","EN_form","EN_label","N_form","N_label","V_form","V_label","J_form"]:
        df[c] = norm_empty(df[c]) if c in df.columns else ""

    df["word_id2"] = pd.to_numeric(df["word_id2"], errors="coerce").astype("Int64")

    for c in ["EN_form","EN_label","N_form","N_label","V_form","V_label"]:
        lab[c] = norm_empty(lab[c])

    lab_emptyN = lab[(lab["N_form"]=="") & (lab["N_label"]=="")]
    lab_nonemptyN = lab[~((lab["N_form"]=="") & (lab["N_label"]==""))]

    df = df.sort_values(["sen_id","word_id2"]).reset_index(drop=True)
    g = df.groupby("sen_id", sort=False)

    # next1
    df["n1_word_id2"] = g["word_id2"].shift(-1)
    df["n1_N_form"]   = norm_empty(g["N_form"].shift(-1))
    df["n1_N_label"]  = norm_empty(g["N_label"].shift(-1))
    df["n1_V_form"]   = norm_empty(g["V_form"].shift(-1))
    df["n1_V_label"]  = norm_empty(g["V_label"].shift(-1))

    # next2
    df["n2_word_id2"] = g["word_id2"].shift(-2)
    df["n2_N_form"]   = norm_empty(g["N_form"].shift(-2))
    df["n2_N_label"]  = norm_empty(g["N_label"].shift(-2))
    df["n2_V_form"]   = norm_empty(g["V_form"].shift(-2))
    df["n2_V_label"]  = norm_empty(g["V_label"].shift(-2))

    # -------- 2-row --------
    cand2 = df[[
        "sen_id","word_id2","EN_form","EN_label","J_form",
        "n1_word_id2","n1_N_form","n1_N_label","n1_V_form","n1_V_label"
    ]].copy()

    cand2 = cand2.rename(columns={
        "word_id2":"EN_word_id2",
        "n1_word_id2":"VX_word_id2",
        "n1_N_form":"N_form",
        "n1_N_label":"N_label",
        "n1_V_form":"V_form",
        "n1_V_label":"V_label",
        "J_form":"EN_J_form",
    })

    # label N empty
    cand2_empty = cand2[(cand2["N_form"]=="") & (cand2["N_label"]=="")]
    m2_empty = cand2_empty.merge(
        lab_emptyN,
        how="left",
        on=["EN_form","EN_label","N_form","N_label","V_form","V_label"]
    )

    # label N non-empty
    cand2_nonempty = cand2[~((cand2["N_form"]=="") & (cand2["N_label"]==""))]
    m2_nonempty = cand2_nonempty.merge(
        lab_nonemptyN,
        how="left",
        on=["EN_form","EN_label","N_form","N_label","V_form","V_label"]
    )

    m2 = pd.concat([m2_empty, m2_nonempty], ignore_index=True)
    m2 = m2[m2["VX_No"].notna()].copy()

    if len(m2):
        m2["VX_No"] = m2["VX_No"].astype(int)
        m2["VX_form"] = [
            make_vx_form_two_row(e, j, n, v)
            for e,j,n,v in zip(m2["EN_form"], m2["EN_J_form"], m2["N_form"], m2["V_form"])
        ]
        m2["match_len"] = 2

    # -------- 3-row (label N non-empty only) --------
    cand3 = df[[
        "sen_id","word_id2","EN_form","EN_label",
        "n1_word_id2","n1_N_form","n1_N_label",
        "n2_word_id2","n2_N_form","n2_N_label","n2_V_form","n2_V_label"
    ]].copy()

    cand3["J_form_mid"] = norm_empty(g["J_form"].shift(-1))

    cand3 = cand3.rename(columns={
        "word_id2":"EN_word_id2",
        "n2_word_id2":"VX_word_id2",
        "n1_N_form":"N_form",
        "n1_N_label":"N_label",
        "n2_V_form":"V_form",
        "n2_V_label":"V_label",
        "n2_N_form":"N_form_last",
        "n2_N_label":"N_label_last",
    })

    cand3 = cand3[((cand3["N_form"]!="") | (cand3["N_label"]!=""))]
    cand3 = cand3[(cand3["N_form_last"]=="") & (cand3["N_label_last"]=="")]

    m3 = cand3.merge(
        lab_nonemptyN,
        how="left",
        on=["EN_form","EN_label","N_form","N_label","V_form","V_label"]
    )
    m3 = m3[m3["VX_No"].notna()].copy()

    if len(m3):
        m3["VX_No"] = m3["VX_No"].astype(int)
        m3["VX_form"] = [
            make_vx_form_three_row(e,n,j,v)
            for e,n,j,v in zip(m3["EN_form"], m3["N_form"], m3["J_form_mid"], m3["V_form"])
        ]
        m3["match_len"] = 3

    edges = pd.concat(
        [m2[["sen_id","EN_word_id2","VX_word_id2","VX_No","VX_form","match_len"]] if len(m2) else pd.DataFrame(),
         m3[["sen_id","EN_word_id2","VX_word_id2","VX_No","VX_form","match_len"]] if len(m3) else pd.DataFrame()],
        ignore_index=True
    )

    if len(edges):
        edges = edges.sort_values(
            ["sen_id","EN_word_id2","match_len","VX_word_id2"],
            ascending=[True,True,False,True]
        ).drop_duplicates(
            ["sen_id","EN_word_id2"], keep="first"
        ).reset_index(drop=True)

        edges["EN_word_id2"] = edges["EN_word_id2"].astype(int)
        edges["VX_word_id2"] = edges["VX_word_id2"].astype(int)

    return edges
#
#③: chain + vx_len 계산
def annotate_word_with_chains(df_word, edges):
    df = df_word.copy()
    df["sen_id"] = norm_empty(df["sen_id"])
    df["word_id2"] = pd.to_numeric(df["word_id2"], errors="coerce").astype("Int64")

    df = df.sort_values(["sen_id","word_id2"]).reset_index(drop=True)

    for c in OUT_COLS_ADDED:
        df[c] = pd.NA

    idx_map = {
        (r.sen_id, int(r.word_id2)): i
        for i, r in df.iterrows()
        if pd.notna(r.word_id2)
    }

    if edges is None or len(edges)==0:
        df["vx_len"] = 0
        return df

    edges = edges.copy()
    edges["sen_id"] = norm_empty(edges["sen_id"])
    edges["EN_word_id2"] = edges["EN_word_id2"].astype(int)
    edges["VX_word_id2"] = edges["VX_word_id2"].astype(int)
    edges["VX_No"] = edges["VX_No"].astype(int)

    for sen, e in edges.groupby("sen_id", sort=False):
        out_map = {a:b for a,b in zip(e["EN_word_id2"], e["VX_word_id2"])}
        no_map  = {a:n for a,n in zip(e["EN_word_id2"], e["VX_No"])}
        form_map= {a:f for a,f in zip(e["EN_word_id2"], e["VX_form"])}

        incoming = set(out_map.values())
        roots = [n for n in out_map if n not in incoming] or list(out_map)

        vx_len_memo = {}
        final_memo = {}

        def compute(node):
            if node in vx_len_memo:
                return vx_len_memo[node]
            cur=node; seen=set(); length=0; last=node
            while cur in out_map and cur not in seen:
                seen.add(cur)
                cur = out_map[cur]
                length += 1
                last = cur
            vx_len_memo[node] = length
            final_memo[node] = last
            return length

        for n in set(out_map)|set(out_map.values()):
            compute(n)

        # EN rows
        for en, vx in out_map.items():
            k=(sen,en)
            if k in idx_map:
                i=idx_map[k]
                df.at[i,"Next_VX_No"]=no_map[en]
                df.at[i,"Next_VX_No_form"]=form_map[en]
                df.at[i,"vx_len"]=vx_len_memo.get(en,0)
                df.at[i,"f_word_id"]=final_memo.get(en,en)

        # VX rows
        for root in roots:
            cur=root; order=1; seen=set()
            while cur in out_map and cur not in seen:
                seen.add(cur)
                vx=out_map[cur]

                if (sen,vx) in idx_map:
                    j=idx_map[(sen,vx)]
                    df.at[j,"vx0_No"]=no_map[cur]
                    df.at[j,"vx0_form"]=form_map[cur]
                    df.at[j,"vx0_order"]=order
                    df.at[j,"V_word_id"]=root
                    df.at[j,"vx_len"]=vx_len_memo.get(vx,0)
                    df.at[j,"f_word_id"]=final_memo.get(vx,vx)

                if (sen,cur) in idx_map:
                    df.at[idx_map[(sen,cur)],"V_word_id"]=root

                cur=vx; order+=1

    df["vx_len"] = pd.to_numeric(df["vx_len"], errors="coerce").fillna(0).astype(int)
    for c in ["Next_VX_No","vx0_No","vx0_order","V_word_id","f_word_id"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    return df


In [ ]:
# 파일 경로
WORD_CSV = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver7.csv"
LABEL_CSV = r"..\..\label\000.vx_label(2026).csv"
# 데이터 불러오기
df_word = pd.read_csv(WORD_CSV)
df_label = pd.read_csv(LABEL_CSV)

# 원본 유지 열만 사용
df_base = df_word[[c for c in KEEP_COLS if c in df_word.columns]].copy()

edges = build_vx_edges_word_only(df_base, df_label)
print("edges:", len(edges))
display(edges.head(20))

df_out = annotate_word_with_chains(df_base, edges)
display(df_out[df_out["vx_len"]>0].head(20))
edges.to_csv(r"..\csv\result\세종문어_vx_edges_long.csv", index=False, encoding="utf-8-sig")


In [ ]:
### 후처리
df = df_out.copy()

# 누락값 채움 & category 변환
#%% 누락값 채움. 
#object를 category형으로 바꿈. : 45초 걸림

exclude_cols = ['form/label', 'N_form', 'V_form']  # 제외할 열

for col in df.columns:
    if col in exclude_cols:
        continue  # 제외할 열은 건너뛰기

    # 숫자형이면 -1로 채움
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(-1)

    # 문자열(object)이면 'NULL'로 채우고 고유값 적을 때만 category로
    elif pd.api.types.is_object_dtype(df[col]):
        df[col] = df[col].fillna('NULL')
        unique_ratio = df[col].nunique(dropna=False) / len(df)
        if unique_ratio < 0.5:
            df[col] = df[col].astype('category')

# EP정보 덧붙이기
if True:
    # --- EP_form & flags ---
    if 'EP_form' in df.columns:
        s = df['EP_form'].astype('string').str.replace(' + ', '', regex=False)
        df['EP_form'] = s

#category 업데이트
# --- file_id → new category 적용 --- 
if True:
    mapping_df = pd.read_csv('../label/TAG_file_Id_new_category.csv')
    id_to_category = dict(zip(mapping_df['file_id'], mapping_df['category']))

    # --- 1. docu_id → file_id로 통일 ---
    if 'file_id' not in df.columns and 'docu_id' in df.columns:
        df = df.rename(columns={'docu_id': 'file_id'})

    # --- 2. category 열이 없으면 생성 ---
    if 'category' not in df.columns:
        df['category'] = pd.NA

    # --- 3. file_id 기준으로 category 매핑 ---
    if 'file_id' in df.columns:
        df['category'] = (
            df['file_id']
            .map(id_to_category)
            .fillna(df['category'])
        )


In [ ]:
#후처리2 - 문장 끝 V 표시(V_label 또는 EN_label 기준)

import pandas as pd

def has_value(s: pd.Series) -> pd.Series:
    """NaN / 빈문자열 / 'NULL' 제외한 값 여부"""
    return (
        s.notna()
        & s.astype(str).str.strip().ne("")
        & s.astype(str).str.upper().ne("NULL")
    )

# 1) V 여부 마스크: V_label OR EN_label
v_mask = has_value(df["V_label"]) | has_value(df["EN_label"])

# 2) sen_id별 마지막 V 인덱스
last_v_idx = df.loc[v_mask].groupby("sen_id").tail(1).index

# 3) 표시 컬럼
df["sent_end_V"] = False
df.loc[last_v_idx, "sent_end_V"] = True



In [ ]:
df.to_csv(r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.csv", index=False, encoding="utf-8-sig")